# Exploratory Data Analysis on claims.csv dataset

|**OMOHL**|**DESCRIPTION**|
|----------|-----------------------------------------|
|**ORIGIN**| This dataset was obtained via web-scraping on the page colombiacheck from 2018-10-26 to 2026-07-16|
|**MOTIVATION**| This project exist because of misinformation and polarization related-problems within colombian society, the main focus is on train a transformer-based model such as BETO to train on the dataset and been able to understand the labeling done to replicate in future news the label |
|**OBJECTIVE**|Curate this dataset, removing missing values and make decisions of data preprocessing to pass into a nlp model |
|**HYPOTHESIS**|N/A |
|**LIMITATIONS**|The data obtained only constitus one dataset from one web page, the bias and limitations that colombiacheck has are included within this analysis and corpus. The model to be trained does not verify facts, it only reproduces the jugdment to classify news given an organization |

- **dataset features**

        - URL: url of the article being reviewed - str
        - verdict: label being given after reviewed [Falso, Cuestionable, Verdadero, Verdadero pero] - str
        - claim_reviewed: corpus to evaluate - str
        - rating: label given by the claimreviewed - str
        - pub_date: publication data - date
        - has_claimreview: boolen whether the new has reviewed - boolean


# 1. DATA ACQUISITION & INITIAL INSPECTION

In [ ]:
import matplotlib.pyplot as plt
import missingno as msno
import numpy as np
import pandas as pd
import seaborn as sns

from fake_news_co.paths import CLAIMS_CSV # Export the path to the raw data directory

In [ ]:
# Load the claims dataset

df = pd.read_csv(CLAIMS_CSV) #load de data
df = df.drop(columns=["jsonld_types"]) # Drop the "jsonld_types" column as it is not needed for analysis

col_rename = {
    "claim_reviewed": "corpus",
    "pub_date": "date",
    "has_claimreview": "has_review"
}

df = df.rename(columns=col_rename) # Rename columns for clarity
df["rating"] = df["rating"].replace({
    "Verdadero pero...": "Verdadero pero"} ) # Standardize the "rating" column values for consistency

df["date"] = pd.to_datetime(df["date"], errors="coerce") # Convert the "date" column to datetime format, coercing errors to NaT

shape = df.shape
print(f"The dataset has {shape[0]} rows and {shape[1]} columns.")
print("-----"*20)

print("\nColumns in the dataset:")
print(df.columns)
print("-----"*20)


print("\nData types of each column:")
print(df.dtypes)
print("-----"*20)

duplicates = df.duplicated().sum()
print(f"\nNumber of duplicate rows in the dataset: {duplicates}")
print("-----"*20)

print(df.info()) # Display a concise summary of the DataFrame, including the number of non-null entries and data types

# 2. MISSING VALUES ANALYSIS

In [ ]:
# Count and percentage
missing = pd.DataFrame({
    'count':   df.isna().sum(),
    'percent': df.isna().mean().mul(100).round(2)
}).sort_values('percent', ascending=False)
print(missing[missing['count'] > 0])

# Visual pattern — which rows and features are missing together?
msno.matrix(df)      # white = missing, black = present
msno.heatmap(df)     # correlation between missingness across features
plt.show()

**Missingness here is structural, not random (MNAR).** The empty `corpus` values are not a data-quality problem: `corpus`, `rating` and `date` go missing *together* and **exactly** when `has_review == False` — i.e. the article carries no ClaimReview markup (older articles and the `Chequeo Múltiple` format). This accounts for 1,815 rows (38.16%).

These rows **must** be dropped — a model cannot learn from an empty claim — but dropping them is a **non-random selection**, not neutral cleaning. It keeps only articles that happen to expose ClaimReview markup, which skews the corpus toward recent articles and toward `Falso`. The next cell quantifies how much this selection costs each class; this bias is recorded in the Data Statement.

In [ ]:
# Selection: keep only rows that have a claim (ClaimReview markup present).
# Required (no model input without a claim) but NON-random, so first quantify
# how the selection reshapes each class before applying it.

before = df["verdict"].value_counts()
kept = df.dropna(subset=["corpus"])["verdict"].value_counts()
retention = pd.DataFrame({
    "before": before,
    "after":  kept.reindex(before.index).fillna(0).astype(int),
})
retention["retention_%"] = (retention["after"] / retention["before"] * 100).round(1)
print("Per-class effect of requiring a non-empty corpus (ClaimReview selection):")
print(retention)
print("-----" * 20)

# Apply the selection.
df = df.dropna(subset=["corpus"]).copy() # Drop rows where "corpus" is NaN

# De-duplicate on the claim text: the same claim in two rows would leak across
# a train/test split and inflate metrics (duplicates verified label-consistent).
n_dups = df["corpus"].duplicated().sum()
df = df.drop_duplicates(subset=["corpus"]).copy()
print(f"Rows after selection: {len(df) + n_dups}  ->  after de-duplicating corpus: {len(df)}  ({n_dups} duplicate claims removed)")

msno.matrix(df, labels=True)     # white = missing, black = present
plt.title("Missing Values After Selecting Rows With a Claim", fontsize=14, fontweight="bold", pad=15)
plt.show()

if df.isna().sum().sum() > 0:
    missing_after = pd.DataFrame({
        'count':   df.isna().sum(),
        'percent': df.isna().mean().mul(100).round(2)
    }).sort_values('percent', ascending=False)
    print(missing_after[missing_after['count'] > 0])
else:
    print("\nNo missing values remain after the selection.")

# 3. UNIVARIATE ANALYSIS

In [ ]:
# Organice and set the index of the DataFrame for time series analysis

df = df.sort_values(by="date", ascending=True) # Sort the DataFrame by the "date" column for chronological analysis
df = df.set_index("date") # Set the "date" column as the index for time series analysis

In [ ]:
# Date range of the dataset
min_date = df.index.min()
max_date = df.index.max()
print(f"\nThe dataset covers a date range from {min_date.date()} to {max_date.date()}.")

fig, ax = plt.subplots(1,2,figsize=(14, 6))
plt.suptitle("Number of Claims Over Time", fontsize=16, fontweight="bold", y=1.05)
sns.countplot(data=df, x=df.index.year, palette="viridis", ax=ax[0], hue=df.index.year, legend=False) # Create a count plot of the number of claims per year
bars = ax[0].patches
for bar in bars:
    height = bar.get_height()
    ax[0].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8) # Annotate each bar with its height
ax[0].set_title("Number of Claims per Year", fontsize=10, fontweight="bold") # Set the title for the first subplot
ax[0].set_xlabel("")
ax[0].set_ylabel("Number of Claims", fontsize=10, fontweight="bold")
sns.countplot(data=df, x=df.index.month, palette="viridis", ax=ax[1], hue=df.index.month, legend=False) # Create a count plot of the number of claims per month
bars = ax[1].patches
for bar in bars:
    height = bar.get_height()
    ax[1].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height), xytext=(0, 3), textcoords="offset points", ha='center', va='bottom', fontsize=8) # Annotate each bar with its height
ax[1].set_title("Number of Claims per Month", fontsize=10, fontweight="bold") # Set the title for the second subplot
ax[1].set_xlabel("")
ax[1].set_ylabel("")
sns.despine(ax=ax[0]) 
sns.despine(ax=ax[1])
plt.tight_layout() # Adjust the layout to prevent overlap
plt.show()

- Even though the minimum date is from 2018, one can see it only got 6 claims reviewed in 2018 and 3 claims on 2019. 

- Interesting enough the claims quota is lower in January and December compared to others month.

- In 2026 (mid-year currently) only 195 claims reviewed, if we extrapolate the average claims for the remaining months we got an approximate value of 407 claims reviewd; showing the claims quota has decreased since 2025.

In [ ]:
# Visualize the distribution of the "verdict" vs "rating" columns

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
plt.suptitle("Distribution of Verdict and Rating", fontsize=16, fontweight="bold", y=1.05)
sns.countplot(x="verdict", data=df, order=df["verdict"].value_counts().index, ax=ax[0], palette="pastel", hue="verdict")
bars = ax[0].patches
for bar in bars:
    height = bar.get_height()
    ax[0].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=10, fontweight='bold')
sns.countplot(x="rating", data=df, order=df["rating"].value_counts().index, ax=ax[1], palette="pastel", hue="rating")
bars = ax[1].patches
for bar in bars:
    height = bar.get_height()
    ax[1].annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3), textcoords="offset points",
                   ha='center', va='bottom', fontsize=10, fontweight='bold')
ax[0].set_title("Distribution of Verdict")
sns.despine(ax=ax[0])
sns.despine(ax=ax[1])
ax[0].set_xlabel("")
ax[1].set_xlabel("")
ax[1].set_title("Distribution of Rating")
plt.show()

- There are discrepancies between the `verdict` (ColombiaCheck listing) and `rating` (ClaimReview) labels, explored in section 4.

- There is a strong class imbalance toward `Falso`. This makes the metric choice critical: **macro-F1** (equal weight to every class) rather than accuracy.

- The task is framed as **multi-class, single-label** classification with 3 categories: `[Falso, Cuestionable, Verdadero]`.

- `Verdadero pero` is a nuanced label; in this methodology it is merged into `Verdadero`, which also helps by increasing that class's (very small) support.

# 4. BIVARIATE & MULTIVARIATE ANALYSIS

In [ ]:
# Verdadero pero merging

df["rating"] = df["rating"].replace({
    "Verdadero pero": "Verdadero"
})
df["rating"].value_counts().plot(kind="bar", color="skyblue", figsize=(10, 6))
bars = plt.gca().patches
for bar in bars:
    height = bar.get_height()
    plt.gca().annotate(f'{height}', xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords="offset points",
                       ha='center', va='bottom', fontsize=10, fontweight='bold')
plt.title("Distribution of Ratings After Merging 'Verdadero pero' into 'Verdadero'", fontsize=16, fontweight="bold")
plt.xlabel("Rating", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Discrepancies between "verdict" and "rating" columns

discrepancies = df[df["verdict"] != df["rating"]]
shape = discrepancies.shape
print(f"\nNumber of discrepancies between 'verdict' and 'rating': {shape[0]}")
print("-----"*20)

print("\nDiscrepancies between 'verdict' and 'rating':")
print(discrepancies[["verdict", "rating"]])
print("-----"*20)

print("\nCorpus of discrepancies between 'verdict' and 'rating':")
print(discrepancies["corpus"])

- The 6 discrepancies were manually reviewed via their URLs. In **all 6**, the URL slug and article confirm the `rating` (e.g. `es-falso-que...`, `falsa-relacion...`, `no-fueron...`); it is `verdict` that is wrong.

- **Root cause:** `verdict` is derived from the listing card text by matching the first known label in a fixed order (`Verdadero` precedes `Falso`), so any headline that contains the word *"verdadero"* is mislabeled. `rating` comes from the structured ClaimReview markup and is complete (0 nulls on the selected corpus).

- **Decision:** use `rating` as the training label.